# 06 - Distributed Computing Concepts: Lazy Evaluation, DAG, Stage Metrics & Persistence Strategy

Fills the remaining Technical Requirement gaps from the brief:
- Demonstrates and explains **lazy evaluation** and the **DAG** (directed acyclic graph)
- Captures and reports **stage metrics** (task counts, shuffle behaviour)
- Explains the **data persistence strategy** used across this project (intermediate writes vs. checkpointing)

**Run this AFTER `02_PySpark_Load_Clean_Partition.ipynb`** -- it reads the same flattened CSVs.

**For your report:** take a Spark UI screenshot (http://localhost:4040) while Section 3's action cell is running -- this notebook is designed to keep a job active long enough to capture stage detail clearly.


## 1. Start Spark session

In [ ]:
import os
os.environ["PYSPARK_PYTHON"] = "python"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time

spark = SparkSession.builder \
    .appName("DistributedComputingConcepts") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

sc = spark.sparkContext
print("Application ID:", sc.applicationId)
print("Spark UI available at: http://localhost:4040")

## 2. Lazy evaluation demonstration

PySpark does not execute transformations (`.filter()`, `.groupBy()`, `.agg()`, etc.) immediately -- it builds a **logical plan** and only executes when an **action** (`.count()`, `.show()`, `.collect()`, `.write()`) is called. This is lazy evaluation.

Below, several transformations are chained together. Calling `.explain()` shows the plan Spark has built *without running anything yet* -- proving no computation has occurred.

In [ ]:
tt = spark.read.csv(r"D:\Dataset\timetables_flat.csv", header=True, inferSchema=True)

# Chain of transformations -- none of this executes yet (lazy)
transformed = (tt
    .filter(F.col("stop_sequence") > 0)
    .groupBy("line_name")
    .agg(F.count("*").alias("stop_time_count"),
         F.countDistinct("vehicle_journey_code").alias("trip_count"))
    .orderBy(F.desc("trip_count")))

print("Transformations defined. Nothing has executed yet -- this is lazy evaluation.")
print("\nLogical/physical plan (the DAG Spark has built so far):")
transformed.explain(extended=False)

## 3. Triggering an action -- this is when the DAG actually executes

Calling `.count()` below is an **action**. Only now does Spark schedule the DAG into stages and tasks and actually run them across partitions.

In [ ]:
start = time.time()
result_count = transformed.count()
elapsed = time.time() - start

print(f"Action triggered ('.count()'). Result: {result_count} routes.")
print(f"Time taken: {elapsed:.2f} seconds")
print("\nWhile this cell was running, the DAG execution could be observed live in the Spark UI (localhost:4040 -> Jobs / Stages tab).")

## 4. Capturing stage metrics programmatically

Rather than relying only on the Spark UI screenshot, this cell pulls stage-level metrics directly via `SparkContext.statusTracker()` -- task counts, completion status -- for the job just run. This confirms the 8-partition configuration is reflected in actual task counts per stage.

In [ ]:
status = sc.statusTracker()
job_ids = status.getJobIdsForGroup()

print("Recent job IDs:", job_ids[:5])
print()

for jid in job_ids[:3]:
    job_info = status.getJobInfo(jid)
    if job_info:
        stage_ids = list(job_info.stageIds)
        print(f"Job {jid} (status={job_info.status}) -- stages: {stage_ids}")
        for sid in stage_ids:
            stage_info = status.getStageInfo(sid)
            if stage_info:
                print(f"  Stage {sid}: numTasks={stage_info.numTasks}, "
                      f"completedTasks={stage_info.numCompletedTasks}, "
                      f"activeTasks={stage_info.numActiveTasks}")
        print()

## 5. Partitioning, caching, and shuffle -- observing the effect directly

Repartitioning to 8 partitions and caching (as done in Notebook 02) changes both the number of tasks per stage and whether a stage needs a shuffle (data movement across partitions). This cell repeats that setup here and confirms the partition count is reflected in the resulting stage's task count -- direct evidence for the report's Big Data Evidence section.

In [ ]:
tt_repartitioned = tt.repartition(8)
tt_repartitioned.cache()
tt_repartitioned.count()  # materialize the cache with an action

print("Number of partitions after repartition(8):", tt_repartitioned.rdd.getNumPartitions())

# A groupBy triggers a shuffle (Exchange) -- visible in the physical plan below
grouped = tt_repartitioned.groupBy("line_name").agg(F.count("*").alias("cnt"))
grouped.explain()
print("\nNote the 'Exchange hashpartitioning' step in the plan above -- this is the shuffle caused by groupBy,")
print("redistributing data across partitions by key before aggregating.")

## 6. Data persistence strategy used in this project

This project uses **intermediate writes** rather than Spark checkpointing, for the following reasons:

- **Between notebooks**, cleaned data is written to CSV/Parquet (`D:\Dataset\output\...`) after each processing stage (Notebook 02's cleaning, Notebook 03's join). This breaks the pipeline into independently re-runnable stages -- if Notebook 04 or 05 needs to be re-run, it does not need to re-parse the raw XML or re-run the full PySpark cleaning job from scratch.
- **`.cache()`** is used within a single notebook session (as in Section 5 above, and in Notebook 02) to keep a DataFrame in memory across multiple actions within that session, avoiding recomputation of the same transformations repeatedly.
- **Spark checkpointing** (`.checkpoint()`, which truncates lineage by writing to reliable storage such as HDFS) was not used here, since this project runs on a single local machine rather than a distributed cluster, and the DataFrame lineage in this pipeline is short enough that lineage-related performance issues did not arise. Checkpointing is more valuable in long, iterative pipelines (e.g. graph algorithms, iterative ML training) where lineage can grow very deep across many transformations -- not the case in this project's linear ETL-style pipeline.

**Trade-off documented for the report:** intermediate writes to disk cost some time/storage compared to keeping everything in memory across notebooks, but they provide resilience (a crash or Windows/Java issue during any one stage, such as the winutils configuration problems encountered in Notebook 02, does not require the entire pipeline to be re-run from the beginning) and make each notebook's logic independently testable and auditable -- both useful properties for a coursework project assessed stage-by-stage.

In [ ]:
spark.stop()
print("Spark session stopped.")